In [1]:
# MT Adaptation with SmolDoc: FULL SFT on H100 (DeepSeek-R1-0528-Qwen3-8B)
import os
import torch
from tqdm import tqdm

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from trl import SFTTrainer, SFTConfig

from utils import get_extended_datasets

torch.set_float32_matmul_precision("high")  # helps on H100

# --- CONFIG ---

SMOLDOC_CONFIG = "smoldoc__en_sw"

# Base model: DeepSeek R1 distilled into Qwen3 8B arch
SFT_MODEL_NAME = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"

OUTPUT_DIR_SFT = f"checkpoints/sft_{SMOLDOC_CONFIG}"
os.makedirs("checkpoints", exist_ok=True)

print("Config:", SMOLDOC_CONFIG)
print("SFT base model:", SFT_MODEL_NAME)

# --- Load SmolDoc + factuality annotations, pick one config, build text field ---

datasets = get_extended_datasets(save_path="../../data/smoldoc_datasets", overwrite=False)
ds_full: Dataset = datasets[SMOLDOC_CONFIG]

print(ds_full)

# Split: 90% for training, 10% held-out test
split_1 = ds_full.train_test_split(test_size=0.1, seed=42)
train_eval_ds = split_1["train"]  # This is the 90%
test_ds = split_1["test"]  # This is the 10% held-out

# Further split the 90% into 80% train and 20% eval (0.8 * 0.9 = 0.72 total, 0.2 * 0.9 = 0.18 total)
split_2 = train_eval_ds.train_test_split(test_size=0.2, seed=42)
train_ds = split_2["train"]  # 72% of original data
eval_ds = split_2["test"]    # 18% of original data

print(f"Train size: {len(train_ds)} ({len(train_ds)/len(ds_full)*100:.1f}% of total)")
print(f"Eval size: {len(eval_ds)} ({len(eval_ds)/len(ds_full)*100:.1f}% of total)")
print(f"Test size: {len(test_ds)} ({len(test_ds)/len(ds_full)*100:.1f}% of total)")

Config: smoldoc__en_sw
SFT base model: deepseek-ai/DeepSeek-R1-0528-Qwen3-8B
📂 Found existing SmolDoc DatasetDict at data/smoldoc_datasets, loading from disk...
📂 Loaded DatasetDict from data/smoldoc_datasets with 102 configs.
Using cached file: data/smoldoc-factuality-ratings.json
Dataset({
    features: ['id', 'sl', 'tl', 'srcs', 'trgs', 'factuality', 'is_src_orig', 'annotator_1_label', 'annotator_1_notes', 'annotator_2_label', 'annotator_2_notes', 'annotator_3_label', 'annotator_3_notes'],
    num_rows: 584
})
Train size: 420 (71.9% of total)
Eval size: 105 (18.0% of total)
Test size: 59 (10.1% of total)


In [2]:
def format_mt_example(srcs, trgs, lang_name="Swahili"):
    """
    Build a single text sequence of the form:

    You are an expert in English to Swahili translation.
    Translate the following English text into Swahili.

    English:
    <src>

    Swahili:
    <tgt>
    """
    src = " ".join(srcs).strip()
    tgt = " ".join(trgs).strip()
    return (
        "You are an expert in English to Swahili translation.\n"
        "Translate the following English text into Swahili.\n\n"
        f"English:\n{src}\n\nSwahili:\n{tgt}"
    )


def add_text_column(batch):
    texts = [
        format_mt_example(srcs, trgs, lang_name="Swahili")
        for srcs, trgs in zip(batch["srcs"], batch["trgs"])
    ]
    return {"text": texts}


def formatting_func(examples):
    # TRL passes a batch as a dict of lists, e.g. {"text": [...], "id": [...]}
    # We just return the text list.
    return examples["text"]


train_ds_fmt = train_ds.map(add_text_column, batched=True)
eval_ds_fmt = eval_ds.map(add_text_column, batched=True)
test_ds_fmt = test_ds.map(add_text_column, batched=True)

print("Example training text:\n", train_ds_fmt[0]["text"][:400])

Example training text:
 You are an expert in English to Swahili translation.
Translate the following English text into Swahili.

English:
The history of Cameroon is long and complex, dating back to the earliest human settlements in the region. The area was first settled by Bantu peoples around 3000 BCE, and it was later conquered by the Kanem-Bornu Empire in the 11th century. In the 15th century, the Portuguese arrived i


In [3]:
# --- Tokenizer ---
tokenizer_sft = AutoTokenizer.from_pretrained(SFT_MODEL_NAME, trust_remote_code=True)
if tokenizer_sft.pad_token is None:
    tokenizer_sft.pad_token = tokenizer_sft.eos_token
tokenizer_sft.padding_side = "right"

# --- SFT Configuration ---
NUM_EPOCHS = 3

# H100 is big, so push the batch size
# Safe starting point for a 4B model @ 2k tokens on 80GB:
per_device_bs = 1        # bump to 8 if it fits
grad_accum = 4           # so global batch = 4 * 1 = 4 sequences

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR_SFT,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=per_device_bs,
    per_device_eval_batch_size=per_device_bs,
    gradient_accumulation_steps=grad_accum,
    learning_rate=5e-5,
    max_length=1024,              # Fixed: was max_length
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=200,
    save_total_limit=2,               # Keep only best 2 checkpoints
    load_best_model_at_end=True,      # Load best model at end
    metric_for_best_model="loss",     # Use validation loss as metric
    bf16=True,                        # H100 loves bf16
    packing=False,                    # can try True later for better throughput
    dataloader_num_workers=4,         # use your CPU a bit
    gradient_checkpointing=False,     # you *can* turn this on, but you have VRAM
    optim="adamw_torch_fused",        # good on NVIDIA GPUs
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
)

# --- Initialize Model ---
print("\nLoading model...")
model_sft = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation="flash_attention_2",
)
model_sft.config.use_cache = False  # Required for gradient checkpointing

print("Model loaded successfully!")

# --- Create Trainer ---
trainer_sft = SFTTrainer(
    model=model_sft,
    processing_class=tokenizer_sft,
    train_dataset=train_ds_fmt,
    eval_dataset=eval_ds_fmt,
    formatting_func=formatting_func,
    args=sft_config,
)

# --- Train ---
print("\n" + "="*50)
print("STARTING TRAINING")
print("="*50 + "\n")

trainer_sft.train()

# --- Save deepseekr1_8b_mt_fullsft model ---
final_dir = os.path.join(OUTPUT_DIR_SFT, "deepseekr1_8b_mt_fullsft")
trainer_sft.save_model(final_dir)
tokenizer_sft.save_pretrained(final_dir)

print(f"\nTraining finished. Model saved to: {final_dir}")


`torch_dtype` is deprecated! Use `dtype` instead!
Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}



Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151645}.



STARTING TRAINING



Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.173300,1.167163,1.120574,133840.000000,0.721192
100,1.054500,1.019388,1.001039,263786.000000,0.752923
150,0.494700,1.034900,0.718653,392542.000000,0.757294
200,0.469600,1.012547,0.691159,524989.000000,0.762631
250,0.265300,1.122711,0.532053,657999.000000,0.758737
300,0.256800,1.120040,0.537750,791323.000000,0.759285



Training finished. Model saved to: checkpoints/sft_smoldoc__en_sw/deepseekr1_8b_mt_fullsft


In [5]:
import re
import evaluate

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR_SFT, "deepseekr1_8b_mt_fullsft")

print("\n\n" + "=" * 50)
print("LOADING MODEL FOR GENERATION-BASED EVALUATION (EVAL SET)")
print("=" * 50)

# --- Load tokenizer ---
ft_tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT_DIR,
    trust_remote_code=True,
)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "left"

# --- Load model ---
ft_model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",   # single H100
    trust_remote_code=True,
)

pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=ft_tokenizer,
    device_map="cuda:0",
)

# --- Strip DeepSeek reasoning if present ---
THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)

def clean_generation(text: str) -> str:
    text = THINK_RE.sub("", text).strip()
    # Keep only first line for MT consistency
    return text.split("\n")[0].strip()

print("\n\n" + "=" * 50)
print("RUNNING EVALUATION ON FULL EVAL SET")
print("=" * 50)

all_predictions = []
all_references = []

# Toggle this if you want to print every example
PRINT_EXAMPLES = True

for i in tqdm(range(len(test_ds)), desc="Evaluating"):
    example = test_ds[i]
    src_text = " ".join(example["srcs"]).strip()
    ref_text = " ".join(example["trgs"]).strip()

    prompt = (
        "You are an expert in English to Swahili translation.\n"
        "Translate the following English text into Swahili.\n\n"
        f"English:\n{src_text}\n\nSwahili:\n"
    )

    out = pipe(
        prompt,
        max_new_tokens=1024,
        do_sample=False,
        return_full_text=False,
    )

    raw_gen = out[0]["generated_text"]
    generated_text = clean_generation(raw_gen)

    all_predictions.append(generated_text)
    all_references.append([ref_text])  # BLEU expects list[list[str]]

    if PRINT_EXAMPLES:
        print(f"\n--- Example {i+1} / {len(test_ds)} ---")
        print(f"SOURCE (EN): {src_text}")
        print(f"REFERENCE (SW): {ref_text}")
        print(f"MODEL OUTPUT (SW): {generated_text}")

# --- BLEU computation ---
bleu_metric = evaluate.load("bleu")
bleu_result = bleu_metric.compute(
    predictions=all_predictions,
    references=all_references,
)

print("\n\n" + "=" * 50)
print("EVAL SET BLEU SCORE")
print("=" * 50)
print(f"BLEU:            {bleu_result['bleu']:.4f}")
print(f"Precisions:      {bleu_result['precisions']}")
print(f"Brevity penalty: {bleu_result['brevity_penalty']:.4f}")
print(f"Length ratio:    {bleu_result['length_ratio']:.4f}")

# End of evaluation block

Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}




LOADING MODEL FOR GENERATION-BASED EVALUATION (EVAL SET)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0




RUNNING EVALUATION ON FULL EVAL SET


...

Evaluating:  98%|██████████████████████████████▍| 58/59 [10:11<00:13, 13.65s/it]


--- Example 58 / 59 ---
SOURCE (EN): Cork flooring is a popular choice for many homeowners because it is durable, stylish, and easy to maintain. However, it is important to know how to properly care for cork flooring in order to extend its lifespan and keep it looking its best. One of the most important things you can do to care for your cork flooring is to sweep, dust, or vacuum it regularly. This will help to remove dirt and debris that can build up on the surface of the flooring and cause it to become damaged. If you have pets, it is especially important to vacuum your cork flooring regularly to remove pet hair and dander. In addition to sweeping, dusting, or vacuuming, you should also occasionally wipe your cork flooring with a damp mop. This will help to remove any dirt or grime that has been missed by the vacuum cleaner. However, it is important to use a damp mop and not a wet mop, as too much water can damage the cork flooring. If you have any spills on your cork flooring, it i

Evaluating: 100%|███████████████████████████████| 59/59 [10:22<00:00, 10.55s/it]


--- Example 59 / 59 ---
SOURCE (EN): Fundamental rights are the essential freedoms that all human beings are entitled to, regardless of their race, religion, gender, sexual orientation, or any other status. These rights are enshrined in the Universal Declaration of Human Rights, which was adopted by the United Nations General Assembly in 1948. The declaration sets out a broad range of fundamental rights, including the right to life, liberty, and security of person; the right to freedom of expression, assembly, and association; the right to freedom of religion; and the right to property. These fundamental rights are essential for the protection of human dignity and the promotion of human flourishing. They are the foundation of a just and equitable society, and they are essential for the maintenance of peace and security. The protection of fundamental rights is a never-ending task. There are always those who would seek to deny or undermine these rights, and we must always be vigilant in



EVAL SET BLEU SCORE
BLEU:            0.2558
Precisions:      [0.5688491393857577, 0.3189180022046977, 0.19439236407022328, 0.12137044967880085]
Brevity penalty: 1.0000
Length ratio:    1.0426
